<a href="https://colab.research.google.com/github/CHAI99k/UTS-BIG_DATA/blob/main/UTS-BIG_DATA_Aplikasi_SIGNAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install google-play-scraper

In [ ]:
from google_play_scraper import reviews, Sort
import csv

result, _ = reviews(
    'app.signal.id',
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=100,
    filter_score_with=None
)

filename = 'ulasan_google_play.csv'


with open(filename, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['userName', 'score', 'at', 'content'])
    writer.writeheader()
    for review in result:

        writer.writerow({
            'userName': review['userName'],
            'score': review['score'],
            'at': review['at'],
            'content': review['content']
        })

print(f"Berhasil menyimpan {len(result)} ulasan ke '{filename}'")

In [ ]:
pip install transformers torch pandas

In [ ]:
import pandas as pd
from transformers import pipeline

# Load the dataset
df = pd.read_csv('ulasan_google_play.csv')

# Initialize the sentiment analysis pipeline
# Note: w11wo/indonesian-roberta-base-sentiment-classifier is a common choice for this task
model_name = "w11wo/indonesian-roberta-base-sentiment-classifier"
sentiment_analyzer = pipeline("sentiment-analysis", model=model_name)

def get_sentiment(text):
    try:
        # Truncate text to fit model limits if necessary
        result = sentiment_analyzer(text[:512])[0]
        return result['label']
    except:
        return "Error"

# Apply analysis
print("Analyzing sentiment...")
df['sentiment'] = df['content'].apply(get_sentiment)

# Display results
display(df[['content', 'sentiment']].head(10))

# Show distribution
print("\nSentiment Distribution:")
print(df['sentiment'].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the visual style
sns.set_theme(style="whitegrid")

# Create the figure
plt.figure(figsize=(10, 6))

# Calculate counts
sentiment_counts = df['sentiment'].value_counts().reset_index()
sentiment_counts.columns = ['Sentiment', 'Total']

# Define custom colors for each sentiment
custom_palette = {'positive': 'seagreen', 'neutral': 'lightgray', 'negative': 'crimson'}

# Create bar plot
ax = sns.barplot(x='Sentiment', y='Total', data=sentiment_counts, palette=custom_palette, hue='Sentiment', legend=False)

# Add labels and title
plt.title('Distribusi Sentimen Ulasan Google Play (Signal)', fontsize=16, fontweight='bold')
plt.xlabel('Kategori Sentimen', fontsize=12)
plt.ylabel('Jumlah Ulasan', fontsize=12)

# Add data labels on top of bars
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha = 'center', va = 'center',
                xytext = (0, 9),
                textcoords = 'offset points',
                fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()